# Topic Modeling in NLP

This notebook applies topic modeling to the 20 Newsgroups text dataset. The main focus is on implementing the methods, viewing the discovered topics, and comparing the results.

Methods used: **LDA**, **NMF**, and optional **BERTopic**.


## 1. Topic modeling — brief theory

Topic modeling is an **unsupervised NLP technique** that finds common themes in a collection of documents without predefined topic labels. For example, a set of news posts may contain topics about sports, medicine, computers, and politics.

- **Corpus:** collection of text documents.
- **Preprocessing:** cleaning text before modeling.
- **Vectorization:** converting text into numbers.
- **Topic words:** words that help describe each topic.
- **Document-topic weights:** show how strongly a document is related to each topic.

**Topic modeling vs. text classification:** topic modeling discovers themes; classification predicts labels that are already defined.


## 2. Methods used

| Method | Basic idea | Input |
|---|---|---|
| **LDA** | Estimates topic-word and document-topic distributions | Word counts |
| **NMF** | Decomposes a non-negative matrix into document-topic and topic-word weights | TF-IDF |
| **BERTopic (optional)** | Uses text embeddings and clustering, then describes each topic with important words | Raw text |

LDA and NMF are the main methods in this notebook. BERTopic is optional because it may need extra installation time.


## 3. Load the dataset

The 20 Newsgroups dataset contains discussion posts. Four categories are selected to make the discovered topics easier to inspect. The category labels are kept for comparison only; they are not used to train LDA or NMF.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

categories = [
    "comp.graphics",
    "rec.sport.hockey",
    "sci.med",
    "talk.politics.mideast",
]

newsgroups = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes"),   # keep only the message body
    shuffle=True,
    random_state=42,
)

raw_df = pd.DataFrame({
    "text": newsgroups.data,
    "category": [newsgroups.target_names[i] for i in newsgroups.target],
})

# Keep posts with at least 30 words (very short posts carry little topic information)
raw_df = raw_df[raw_df["text"].str.split().str.len() >= 30]

# Take the same number of posts from every newsgroup so the corpus is balanced
n_per_group = min(300, raw_df["category"].value_counts().min())
raw_df = (raw_df.groupby("category")
                .sample(n=n_per_group, random_state=42)
                .sample(frac=1, random_state=42)       # shuffle
                .reset_index(drop=True))

documents = raw_df["text"].tolist()          # raw text
true_labels = raw_df["category"].tolist()    # used ONLY for evaluation later

print("Number of documents:", len(documents))
print(raw_df["category"].value_counts())
print("\n--- Example document (first 400 characters) ---")
print(documents[0][:400])
print("Actual newsgroup:", true_labels[0])

## 4. Preprocess the text

The cleaning function converts text to lowercase and removes email addresses, URLs, punctuation, and extra spaces. Keeping preprocessing simple makes it easier to see how the code works.


In [ ]:
import re

def basic_clean(text):
    text = text.lower()
    text = re.sub(r"\S+@\S+", " ", text)            # remove e-mail addresses
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)           # keep letters only (drops digits/punctuation)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_documents = [basic_clean(doc) for doc in documents]
print("Before:", documents[0][:200].replace("\n", " "))
print("After: ", clean_documents[0][:200])

## 5. Convert text into numerical features

- **CountVectorizer** creates word-count features and is used for LDA.
- **TfidfVectorizer** weights words by their importance across documents and is used for NMF.

The vocabulary settings remove very rare and overly common terms.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.5
)
X_count = count_vectorizer.fit_transform(clean_documents)

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.5
)
X_tfidf = tfidf_vectorizer.fit_transform(clean_documents)

print("Count matrix shape:", X_count.shape)
print("TF-IDF matrix shape:", X_tfidf.shape)
print("Example vocabulary:", count_vectorizer.get_feature_names_out()[:25])

## 6. Helper functions

The function below prints the most important words for each topic. A second helper compares topic assignments with the original dataset categories. Topic numbers are arbitrary, so interpret them using the words and documents rather than the number alone.


In [ ]:
from IPython.display import display

def show_top_words(model, feature_names, n_words=10, method_name="Model"):
    print(f"\n===== {method_name}: top words =====")
    for topic_idx, component in enumerate(model.components_):
        top_indices = component.argsort()[-n_words:][::-1]
        words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: " + ", ".join(words))


def show_label_crosstab(topic_assignments, true_labels, method_name="Model"):
    """Rows = discovered topic, columns = real newsgroup. Large counts concentrated in one cell per row = clean topics."""
    table = pd.crosstab(
        pd.Series(topic_assignments, name="Discovered topic"),
        pd.Series(true_labels, name="Real newsgroup"),
    )
    print(f"\n===== {method_name}: discovered topics vs real newsgroups =====")
    display(table)

# Method 1 — LDA

**Theory:** Latent Dirichlet Allocation (LDA) treats each document as a mixture of topics and each topic as a distribution over words. It is commonly trained on word counts.

**Code:** Set the number of topics, train the model, and print the top words.


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics_lda = 4
lda_model = LatentDirichletAllocation(
    n_components=n_topics_lda,
    max_iter=30,
    random_state=42,
    learning_method="batch"
)
lda_model.fit(X_count)

lda_terms = count_vectorizer.get_feature_names_out()
show_top_words(lda_model, lda_terms, n_words=10, method_name="LDA")

## 8. View LDA document-topic weights

`transform()` returns topic weights for each document. The largest weight is used as the document's dominant topic. The full weights are also displayed for a few documents.


In [ ]:
from IPython.display import display

lda_doc_topics = lda_model.transform(X_count)
lda_topic_cols = [f"LDA_Topic_{i+1}" for i in range(n_topics_lda)]

lda_results = pd.DataFrame(lda_doc_topics, columns=lda_topic_cols)
lda_results.insert(0, "Document", [" ".join(d.split())[:90] + "..." for d in documents])
lda_results.insert(1, "Real_Newsgroup", true_labels)
lda_results["Dominant_Topic"] = lda_doc_topics.argmax(axis=1) + 1

display(lda_results.head(10).round(3))
show_label_crosstab(lda_results["Dominant_Topic"], true_labels, "LDA")

## 9. Plot LDA topic counts

This chart shows how many documents were assigned to each dominant topic.


In [ ]:
import matplotlib.pyplot as plt

lda_results["Dominant_Topic"].value_counts().sort_index().plot(kind="bar")
plt.title("LDA: Documents per Dominant Topic")
plt.xlabel("Topic")
plt.ylabel("Document count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Method 2 — NMF

**Theory:** Non-negative Matrix Factorization (NMF) splits the TF-IDF matrix into document-topic weights and topic-word weights. Unlike LDA, these weights are not probabilities.

**Code:** Train NMF and display the important words for each topic.


In [ ]:
from sklearn.decomposition import NMF

n_topics_nmf = 4
nmf_model = NMF(
    n_components=n_topics_nmf,
    init="nndsvda",
    random_state=42,
    max_iter=500
)
W_nmf = nmf_model.fit_transform(X_tfidf)
H_nmf = nmf_model.components_

tfidf_terms = tfidf_vectorizer.get_feature_names_out()
show_top_words(nmf_model, tfidf_terms, n_words=10, method_name="NMF")

## 11. View NMF document-topic weights

The largest weight gives each document's dominant topic. NMF weights are not automatically normalized to sum to 1.


In [ ]:
nmf_dominant = W_nmf.argmax(axis=1) + 1
nmf_results = pd.DataFrame(W_nmf, columns=[f"NMF_Topic_{i+1}" for i in range(n_topics_nmf)])
nmf_results.insert(0, "Document", [" ".join(d.split())[:90] + "..." for d in documents])
nmf_results.insert(1, "Real_Newsgroup", true_labels)
nmf_results["Dominant_Topic"] = nmf_dominant
display(nmf_results.head(10).round(3))
show_label_crosstab(nmf_results["Dominant_Topic"], true_labels, "NMF")

nmf_results["Dominant_Topic"].value_counts().sort_index().plot(kind="bar")
plt.title("NMF: Documents per Dominant Topic")
plt.xlabel("Topic")
plt.ylabel("Document count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Method 3 — BERTopic (optional)

**Theory:** BERTopic usually creates text embeddings, groups similar documents, and extracts important words to describe each group. It can capture semantic similarity, but it may take longer to install and run. A topic ID of `-1` usually represents outlier documents.

Run the installation and training cells below only if BERTopic is needed and the environment supports it.


In [ ]:
# Optional: install BERTopic and its dependencies.
# Run this cell only if you want to try BERTopic.
%pip -q install bertopic

In [ ]:
# Optional BERTopic training cell. Run after successful installation.
# (The first run downloads a small sentence-embedding model and may take a few minutes on CPU.)
from bertopic import BERTopic
from umap import UMAP

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=42)
bertopic_vectorizer = CountVectorizer(stop_words="english", min_df=2)

bertopic_model = BERTopic(
    language="english",
    umap_model=umap_model,
    vectorizer_model=bertopic_vectorizer,
    min_topic_size=15,
    verbose=False,
)

bertopic_docs = [" ".join(d.split())[:1000] for d in documents]   # tidy whitespace, cap length
bertopic_topics, bertopic_probs = bertopic_model.fit_transform(bertopic_docs)

display(bertopic_model.get_topic_info().head(10))
print("\nExample topic terms:")
for topic_id in sorted(set(bertopic_topics)):
    if topic_id != -1:
        print(f"Topic {topic_id}:", [w for w, _ in bertopic_model.get_topic(topic_id)[:8]])

show_label_crosstab(bertopic_topics, true_labels, "BERTopic (-1 = outliers)")

# Optional interactive visualizations (if supported by the runtime):
# bertopic_model.visualize_topics()
# bertopic_model.visualize_barchart(top_n_topics=10)

# 12. Compare the methods

Compare the top words, document assignments, and topic interpretability. There is no single score that proves a topic model is best; inspect example documents as well.

| Criterion | LDA | NMF | BERTopic |
|---|---|---|---|
| Input | Word counts | TF-IDF | Text and embeddings |
| Topic count | Set beforehand | Set beforehand | Depends on clustering settings |
| Output | Topic probabilities/weights | Non-negative weights | Topics and representative words |
| Runtime | Usually moderate | Usually moderate | Often higher |



## 13. Experiments and observations

1. Change the selected dataset categories and run the notebook again.
2. Try different topic counts, such as 3, 4, and 5, for LDA and NMF.
3. Compare the top words produced by both methods.
4. Read example documents to check whether each topic makes sense.
5. Try unigram features and unigram-plus-bigram features.
6. Write a short conclusion stating which method produced the most understandable topics for this dataset.
